# Keystroke verification with generative models: full experiment suite

This notebook is the **extensible baseline**: every model family and every experiment in the project.
For the focused question of how a GMM compares with the published CMU benchmark, see `gmm_benchmark.ipynb`.

**Data.** CMU benchmark: 51 subjects × 8 sessions × 50 typings of `.tie5Roanl`. One typing = one data point.
Main features: **21 log timings** (11 hold H + 10 keydown-keydown DD). UD is dropped because UD = DD − H exactly.

**Protocol (official, benchmark-comparable).** For each subject:
* train on typings 1-200 (sessions 1-4);
* genuine test = typings 201-400 (sessions 5-8);
* impostor test = the first 5 typings of each of the other 50 subjects.

Score: higher = more genuine. We report the per-subject EER, then its mean, SD and bootstrap 95% CI over
subjects. Models are compared with **paired tests** against **Scaled Manhattan | 21 log**.

**Tuning** uses the validation split only: fit on 1-160, validate on 161-200, impostors = the other
subjects' typings 6-10. We then refit on 1-200 and evaluate on test **once**. Population models never see
the impostor test typings, and they are also evaluated **open-set** (3 folds of 17 subjects; the held-out
fold serves only as impostors). Every fit is checked by `utils.check_no_leak`.

| section | content |
|---|---|
| 1-2 | setup, saving helpers |
| 3 | data loading, checks and figures |
| 4 | model catalogue and grids |
| 5 | experiment 1: tuning on validation |
| 6 | experiment 2: official training and test, plus open-set |
| 7 | experiment 3: enrollment size |
| 8 | experiment 4: template adaptation |
| 9 | experiment 5: speed normalization |
| 10 | model figures |
| 11 | evaluation: tables, paired tests, GMM diagnostics |
| 12 | save everything |

**Run time.** About an hour on a laptop CPU. Set the environment variable `KS_QUICK=1` before starting
Jupyter for a few-minute smoke run (one grid point per family, fewer enrollment sizes). Its output goes to
`outputs/_quick/`.

## 1. Setup

Imports, paths, seeds and run-size settings. Every computation is in `src/`; the notebook only orchestrates.

In [ ]:
import os, sys, json, time
from pathlib import Path
os.environ.setdefault("OMP_NUM_THREADS", "1")   # avoids a known sklearn KMeans/MKL memory-leak warning on Windows

# Make `src` importable whether the notebook runs from the repo root or from notebooks/.
REPO = Path.cwd().resolve()
if REPO.name == "notebooks":
    REPO = REPO.parent
sys.path.insert(0, str(REPO))

import numpy as np
import pandas as pd
import torch
from IPython.display import display, Markdown

from src import utils as U          # data, feature sets, splits, leak check, EER, statistics
from src import models as M         # detectors and Gaussian-family densities
from src import flow as F           # RealNVP normalizing flow
from src import plots as P          # figure builders (they never save)
from src import experiments as E    # model registry, enrollment/adaptation runners, tables
from src.evaluation import ModelSpec, evaluate, evaluate_openset
from src.saving import Saver

SEED = 0
np.random.seed(SEED); torch.manual_seed(SEED)
pd.set_option("display.width", 200, "display.max_columns", 30, "display.precision", 4)

# --- run size: full by default, small smoke run with KS_QUICK=1 ---
QUICK = os.environ.get("KS_QUICK") == "1"
ENROLL_N = (12, 200) if QUICK else E.ENROLL_N            # enrollment sizes for experiments 3 and 5
STRAT_SEEDS = (0,) if QUICK else E.STRAT_SEEDS           # seeds for the stratified enrollment draws
ADAPT_N0 = (12,) if QUICK else (12, 24)                  # initial template sizes for experiment 4
if QUICK:  # keep only the first grid point of every family
    E.FAMILIES = {f: (make, g[:1]) for f, (make, g) in E.FAMILIES.items()}
BASE = E.BASELINE                                        # "Scaled Manhattan"
print("QUICK mode" if QUICK else "full run", "| families:", len(E.FAMILIES))
T0 = time.time()

## 2. Saving helpers

This is the only cell that sets up saving. Every experiment is a *run* with its own folder,
`outputs/<NAME><NNN>_<timestamp>/`, which is never overwritten.

* `start_run(name, config)` writes `config.json` and `env.json` immediately, **before training**.
* `run.capture()` copies printed output and warnings into `log.txt`.
* `run.add_figure` and `run.set_results` keep figures, metrics, scores and models in memory.
* `save_all()` (section 12) writes them with `save_model`, `save_figure` and `save_run`, and appends one
  row per run to `runs_index.csv`.

In [ ]:
saver = Saver(REPO, REPO / "outputs" / ("_quick" if QUICK else ""))
start_run, save_run, save_model, save_figure, save_all = (
    saver.start_run, saver.save_run, saver.save_model, saver.save_figure, saver.save_all)
print("outputs ->", saver.out)

## 3. Data

`load_data` verifies the MD5 checksum, sorts typings chronologically, and adds `typing` = 1..400 per subject.
We build the main feature matrix (log21) and the raw version used for comparison figures.

In [ ]:
df = U.load_data()                        # stops with download instructions if the CSV is missing
X, NAMES = U.make_features(df, "log21")   # main feature set: (20400, 21)
RAW21, RAW_NAMES = U.make_features(df, "raw21")
print(df.shape, "| subjects:", df.subject.nunique(), "| log21 features:", X.shape[1])
print("open-set folds (subject indices):", [f.tolist() for f in U.openset_folds()])

### 3b. Data figures

* **Raw vs log histograms.** The log transform removes most of the right skew.
* **Within- vs between-user spread.** This shows which features separate subjects.
* **Overall (population) Gaussian.** We compare two descriptions of the pooled training data:
  a single pooled Gaussian, and a mixture of the 51 per-subject Gaussians. The pooled Gaussian is too
  smooth; the mixture of users reproduces the multi-modal shape of the population.
* **Per-user Gaussian.** Real typings vs samples from $p(x \mid u)$, with the model's 2-SD ellipse.

In [ ]:
run_data = start_run("data_overview", {
    "model": "none: data figures + overall Gaussian (pooled, and mixture of per-user Ledoit-Wolf Gaussians)",
    "features": "raw21 vs log21", "protocol": "descriptive", "split_short": "train(1-200)",
    "split": "training typings 1-200 of all users only", "n_train": 200, "sampling": "first", "seed": SEED})
with run_data.capture():
    # Per-user Gaussians fitted on training typings only, used for the figures
    user_lw = {u: M.LedoitWolfGaussian().fit(X[U.rows(u, 1, U.TRAIN_END)]) for u in range(U.N_USERS)}
    tr = U.rows(np.arange(U.N_USERS), 1, U.TRAIN_END)
    skew = {"raw21": float(pd.DataFrame(RAW21[tr]).skew().mean()), "log21": float(pd.DataFrame(X[tr]).skew().mean())}
    print("mean feature skewness:", {k: round(v, 2) for k, v in skew.items()})
run_data.add_figure(P.raw_vs_log(RAW21, X, RAW_NAMES), "raw_vs_log_histograms", report=True)
run_data.add_figure(P.within_between(X, NAMES), "within_vs_between_user_spread", report=True)
run_data.add_figure(P.population_samples(X, user_lw, NAMES), "overall_gaussian_samples_vs_pooled", report=True)
run_data.add_figure(P.user_samples(X, user_lw, NAMES, users=[0, 20, 40],
                                   title="Ledoit-Wolf Gaussian p(x|u): real vs samples"), "user_gaussian_samples")
run_data.set_results(metrics={"mean_skewness": skew}, models=user_lw,
                     best_config={"note": "per-user Ledoit-Wolf Gaussians on typings 1-200, used only for figures"})

## 4. Models

All models implement `fit(X) -> self` and `score(X)` (higher = more genuine); the densities also implement
`sample(n)`. For density models the score is the log-likelihood, and we always report the **test
log-likelihood** next to the EER: values in the millions would signal a collapsed fit.

| family | idea | uses other subjects? |
|---|---|---|
| Scaled Manhattan | L1 distance scaled by per-feature mean absolute deviation (benchmark winner) | no |
| Gaussian diag / full / Ledoit-Wolf | one Gaussian per subject; diagonal, ridge-regularised or shrinkage covariance | no |
| Gaussian tied | subject's own mean, covariance pooled over the other subjects | **yes** |
| Gaussian Bayes/MAP | normal-inverse-Wishart prior from the population; posterior-predictive Student-t | **yes** |
| Student-t | robust heavy-tailed Gaussian fitted by EM | no |
| GMM | full-covariance mixture, $K$ = 1..3 by BIC | no |
| Laplace | product of Laplace marginals: the density version of Scaled Manhattan | no |
| RealNVP | small normalizing flow, from scratch or pretrained on the population then fine-tuned | pretrained: **yes** |

Validation grids:

In [ ]:
pd.DataFrame([{"family": f, "population": f in E.POPULATION_FAMILIES, "n_configs": len(g),
               "grid": json.dumps(g) if len(json.dumps(g)) < 110 else json.dumps(g)[:107] + "..."}
              for f, (make, g) in E.FAMILIES.items()])

## 5. Experiment 1: hyperparameter tuning (validation only)

Every grid point is evaluated on the validation split (fit 1-160, genuine 161-200, impostors = others' 6-10).
The lowest mean validation EER wins. For the flow, early stopping on validation also gives the number of
training epochs. We keep the flow's learning curves from this validation run for section 10.
One run folder per family is created (with its `config.json`) **before** tuning.

In [ ]:
RUNS, TUNED, FLOW_HISTORIES = {}, {}, {}
for fam, (make, g) in E.FAMILIES.items():
    pop = fam in E.POPULATION_FAMILIES
    RUNS[fam] = run = start_run(f"official_{fam}", {
        "model": fam, "hyperparameter_grid": g, "selection": "lowest mean validation EER",
        "features": "log21 (11 H + 10 DD, log seconds)", "protocol": "official (Killourhy & Maxion)",
        "split_short": "val->test",
        "split": {"tuning": "fit typings 1-160, genuine 161-200, impostors = others' typings 6-10",
                  "test": "refit 1-200, genuine 201-400, impostors = others' typings 1-5 (250)"},
        "population": None if not pop else (
            "pretraining: users outside the target's fold, typings 11-200 (val) / 6-200 (test)"
            if fam.startswith("RealNVP") else "others' typings 11-200 (val) / 6-200 (test)"),
        "n_train": 200, "sampling": "first", "seed": SEED})
    with run.capture():
        best, table = E.tune_family(X, fam)
        print("chosen:", best)
        if fam.startswith("RealNVP"):   # re-run the chosen config on validation to keep its learning curves
            vr = evaluate(X, make(**best), "val", keep_models=True)
            FLOW_HISTORIES[fam] = {u: m.history_ for u, m in vr.models.items()}
    run.config["key_settings"] = best
    TUNED[fam] = (best, table)

pd.DataFrame([{"family": f, "chosen": json.dumps(b), "val_mean_eer": t.val_mean_eer[0]} for f, (b, t) in TUNED.items()])

## 6. Experiment 2: training and test

### 6a. Official protocol

Each family is refit on typings 1-200 with its chosen hyperparameters and evaluated **once** on the test typings.

In [ ]:
SPECS, OFFICIAL = {}, {}
for fam, (best, table) in TUNED.items():
    SPECS[fam] = spec = E.final_spec(fam, best, table)   # flow: epochs = median best epoch on validation
    run = RUNS[fam]
    with run.capture():
        res = evaluate(X, spec, "test", keep_models=True)   # the single test evaluation
        s = res.summary()
        print(f"{spec.label():70s} test EER {s['mean_eer']:.4f}  median test LL {s.get('test_ll_median', float('nan')):.2f}")
    OFFICIAL[fam] = res
    run.set_results(per_user=res.per_user, scores=res.scores, models=res.models, grid_table=table,
                    metrics={**s, "chosen": best, "val_mean_eer": float(table.val_mean_eer[0])},
                    best_config={"family": fam, "chosen": best, "final_spec": spec.label()})

### 6b. Open-set evaluation of the population models

In the official protocol the impostors' typings 1-5 come from subjects whose *other* typings may sit in a
population model's training data. Open-set removes that overlap. The subjects are split into 3 folds of 17;
the held-out fold provides the impostors only, and is never used in any population statistic. Each subject
is enrolled in the 2 folds that don't hold it out, and its EER is the mean of those two. The baseline is
re-run under the same impostors, so the paired tests stay valid.

In [ ]:
OPEN = {}
for fam in [BASE, *E.POPULATION_FAMILIES]:
    run = start_run(f"openset_{fam}", {
        "model": fam, "key_settings": TUNED[fam][0], "features": "log21", "protocol": "open-set",
        "split_short": "openset",
        "split": "folds of 17 users; held-out fold = impostors only (their typings 1-5); population = "
                 "non-held-out users' typings 1-200; per-user EER = mean over the 2 folds where the user is enrolled",
        "folds": [f.tolist() for f in U.openset_folds()], "n_train": 200, "sampling": "first", "seed": SEED})
    with run.capture():
        OPEN[fam] = res = evaluate_openset(X, SPECS[fam])
        print(f"{fam:24s} open-set EER {res.eers.mean():.4f}  (official {OFFICIAL[fam].eers.mean():.4f})")
    run.set_results(per_user=res.per_user, metrics=res.summary(), scores=res.scores,
                    best_config={"family": fam, "chosen": TUNED[fam][0],
                                 "note": "hyperparameters from validation; models not kept (3 folds)"})

## 7. Experiment 3: enrollment size

How much enrollment data does each model need? For $n \in \{8, 12, 24, 48, 100, 200\}$:
* **first n**: the subject's first $n$ typings, i.e. one sitting. This is the realistic scenario.
* **stratified**: $n/4$ random typings from each of sessions 1-4, over 5 seeds. Every model sees the same draws.
  This is a diagnostic: it separates *amount of data* from *session drift*.

The test set is always sessions 5-8 vs the others' typings 1-5. The hyperparameters are the ones tuned in section 5.

In [ ]:
ENROLL = []
for fam, spec in SPECS.items():
    run = start_run(f"enroll_{fam}", {
        "model": fam, "key_settings": TUNED[fam][0], "features": "log21",
        "protocol": "official test, varying enrollment", "split_short": "test", "n_train": list(ENROLL_N),
        "sampling": {"first": "typings 1..n", "stratified": f"n/4 random per session 1-4, seeds {list(STRAT_SEEDS)}"},
        "seed": SEED})
    with run.capture():
        t = E.enrollment(X, spec, ns=ENROLL_N, seeds=STRAT_SEEDS).assign(model=fam)
        piv = t.groupby(["sampling", "n"])["eer"].mean().unstack(0)   # rows n, columns first/stratified
        print(fam, "\n", piv.round(4).to_string())
    ENROLL.append(t)
    run.set_results(per_user=t, metrics={"mean_eer_by_n": piv, "mean_eer": float(piv.loc[200, "first"])},
                    best_config={"family": fam, "chosen": TUNED[fam][0]},
                    index={"sampling": "first+stratified", "n_train": f"{min(ENROLL_N)}..{max(ENROLL_N)}"})
ENROLL = pd.concat(ENROLL, ignore_index=True)

## 8. Experiment 4: template adaptation

A realistic deployment enrols a subject once, then updates the template with accepted logins.
* Enroll on the first $n_0$ typings of session 1.
* Replay sessions 2-4 in order, with 50 impostor attempts mixed in (others' typings 11-200, never test typings).
* Every **accepted** attempt is added to the template, and the model is refit every 10 acceptances.
* The threshold is the EER point on validation (the rest of session 1 vs others' typings 6-10), re-chosen
  after each refit.
* Test once on sessions 5-8.

We also count how many **impostor** typings were absorbed into templates. `oracle` = the template
with every genuine session 2-4 typing added (perfect labels).

In [ ]:
ADAPT_FAMS = ([BASE, "Laplace", "Gaussian diag", "Gaussian Ledoit-Wolf", "Gaussian Bayes/MAP", "Student-t", "GMM"]
              if not QUICK else [BASE, "Gaussian diag"])
ADAPT = []
for n0 in ADAPT_N0:
    for fam in ADAPT_FAMS:
        run = start_run(f"adapt_n{n0}_{fam}", {
            "model": fam, "key_settings": {**TUNED[fam][0], "n0": n0}, "features": "log21",
            "protocol": "template adaptation", "split_short": "adapt->test", "n_train": n0,
            "sampling": "first (session 1)",
            "adaptation": {"stream": "sessions 2-4 (150 genuine) + 50 impostor attempts (others' 11-200)",
                           "threshold": "EER point on validation (session 1 rest vs others' 6-10)", "refit_every": 10},
            "seed": SEED})
        with run.capture():
            t, models = E.adaptation(X, SPECS[fam], n0=n0)
            t["model"] = fam
            print(f"n0={n0} {fam:22s}", t[["eer_static", "eer_adapted", "eer_oracle", "impostors_absorbed"]].mean().round(4).to_dict())
        ADAPT.append(t)
        run.set_results(per_user=t, models=models, best_config={"family": fam, "chosen": TUNED[fam][0], "n0": n0},
                        metrics={**U.summarize(t.eer_adapted), "static_mean_eer": float(t.eer_static.mean()),
                                 "oracle_mean_eer": float(t.eer_oracle.mean()),
                                 "adapted_vs_static": U.paired_test(t.eer_adapted, t.eer_static),
                                 "impostors_absorbed_total": int(t.impostors_absorbed.sum()),
                                 "impostor_attempts_total": int(t.impostor_attempts.sum()),
                                 "genuine_accept_rate": float(t.genuine_accepted.sum() / t.genuine_attempts.sum())})
ADAPT = pd.concat(ADAPT, ignore_index=True)

## 9. Experiment 5: speed normalization (first-n enrollment)

People type faster or slower on different days, so the overall speed drifts. We test whether removing it helps:
* **(a) log21**: the 21 log features;
* **(b) rhythm20**: subtract each typing's mean log-time, i.e. keep only the rhythm. The 21 centred values
  sum to zero, which makes the covariance singular, so one feature is dropped, leaving 20;
* **(c) rhythm20+mean**: (b) plus the mean log-time as one extra feature, so speed becomes an explicit dimension.

Each non-flow family is **re-tuned on validation** for each feature set. The flow is omitted here to keep
the run time down.

In [ ]:
SPEED_FEATS = ("log21", "rhythm20", "rhythm20+mean")
SPEED_FAMS = [f for f in E.FAMILIES if not f.startswith("RealNVP")] if not QUICK else [BASE, "Gaussian diag", "GMM"]
SPEED = []
for fs in SPEED_FEATS:
    Xf, _ = U.make_features(df, fs)
    run = start_run(f"speed_{fs}", {"model": SPEED_FAMS, "features": fs, "protocol": "official test, first-n enrollment",
                                    "split_short": "val->test", "selection": "re-tuned on validation per feature set",
                                    "n_train": list(ENROLL_N), "sampling": "first", "seed": SEED})
    chosen = {}
    with run.capture():
        for fam in SPEED_FAMS:
            best, table = E.tune_family(Xf, fam, verbose=False)        # validation only
            chosen[fam] = best
            t = E.enrollment(Xf, E.final_spec(fam, best, table), ns=ENROLL_N, samplings=("first",))
            SPEED.append(t.assign(model=fam, features=fs))
            print(f"{fs:14s} {fam:22s} {json.dumps(best):40s} EER@n=200 {t[t.n == 200].eer.mean():.4f}")
    run.config["key_settings"] = chosen
    d = pd.concat([s for s in SPEED if s.features.iloc[0] == fs])
    run.set_results(per_user=d, best_config={"chosen_per_family": chosen},
                    metrics={"mean_eer_by_model_n": d.groupby(["model", "n"]).eer.mean().unstack(),
                             "mean_eer": float(d[(d.n == 200) & (d.model == BASE)].eer.mean())},
                    index={"model": "all families (mean_eer = baseline at n=200)"})
SPEED = pd.concat(SPEED, ignore_index=True)

## 10. Model figures

* **Samples vs real typings** for every density model, for 3 subjects.
* **Flow learning curves**: train and validation NLL per epoch for the validation run. The validation curve
  turns up early, which is why early stopping matters on about 160 typings.
* **DET curves** (FRR averaged over subjects at fixed FAR) and **per-subject EER bars** against the baseline.
* **Enrollment curves**, **adaptation bars** and **speed-normalization curves** for experiments 3-5.

In [ ]:
run_fig = start_run("figures", {"model": "all", "features": "log21 (+ rhythm sets in experiment 5)", "protocol": "all",
                                "split_short": "test", "n_train": 200, "sampling": "first", "seed": SEED})
# Per-family figures go into that family's own run folder.
for fam, res in OFFICIAL.items():
    run = RUNS[fam]
    if fam != BASE:
        run.add_figure(P.per_user_bars(res.eers, OFFICIAL[BASE].eers, fam, BASE + " | 21 log"), "per_user_eer")
    run.add_figure(P.det_curves({fam: res, BASE: OFFICIAL[BASE]}), "det_curve")
    if getattr(next(iter(res.models.values())), "is_density", False):
        run.add_figure(P.user_samples(X, res.models, NAMES, users=[0, 20, 40],
                                      title=f"{fam} p(x|u): real training typings vs samples"), "user_samples")
for fam, hist in FLOW_HISTORIES.items():
    RUNS[fam].add_figure(P.learning_curves(hist, f"{fam}: learning curves (validation, early stopping)"),
                         "learning_curves", report=fam == "RealNVP")
# Cross-model figures go into the "figures" run.
run_fig.add_figure(P.det_curves(OFFICIAL, "DET, official protocol (log21)"), "det_official_all", report=True)
run_fig.add_figure(P.enrollment_plot(ENROLL, "Enrollment size (log21): first-n vs stratified"), "enrollment_size", report=True)

## 11. Evaluation

### 11a. Official-protocol table

`val_eer` is the validation EER that chose the hyperparameters. `mean_eer`, `sd` and `ci95` are on test.
`diff_vs_base` = mean per-subject (model − Scaled Manhattan), so negative = better. `diff_ci95` is its
bootstrap CI and `wilcoxon_p` the signed-rank test. `better_worse` counts the subjects where the model is
better / worse. `test_ll_*` = mean log-likelihood of the genuine test typings (median / min over subjects).

In [ ]:
official_table = E.paired_table(OFFICIAL, BASE)
official_table.insert(1, "val_eer", official_table.model.map(lambda f: TUNED[f][1].val_mean_eer[0]))
official_table.insert(2, "chosen", official_table.model.map(lambda f: json.dumps(TUNED[f][0])))
run = start_run("table_official", {"model": "all", "features": "log21", "protocol": "official", "split_short": "test",
                                   "baseline": BASE + " | 21 log", "n_train": 200, "sampling": "first", "seed": SEED,
                                   "paired_tests": "bootstrap CI of per-user EER difference + Wilcoxon signed-rank"})
run.set_results(per_user=pd.DataFrame({f: r.eers for f, r in OFFICIAL.items()}).assign(user=U.users_of(df)),
                metrics={"table": official_table}, tables={"official_table": official_table},
                index={"mean_eer": official_table.loc[0, "mean_eer"], "model": "table (best: " + official_table.loc[0, "model"] + ")"})
official_table

### 11b. Open-set table

The same paired comparison, but with open-set impostors. `official_mean_eer` is shown for reference:
if a population model got much worse open-set, it was benefiting from having seen the impostors.

In [ ]:
openset_table = E.paired_table(OPEN, BASE)
openset_table["official_mean_eer"] = openset_table.model.map(lambda f: OFFICIAL[f].eers.mean())
run = start_run("table_openset", {"model": "population models", "features": "log21", "protocol": "open-set",
                                  "split_short": "openset", "baseline": BASE + " | 21 log (open-set impostors)",
                                  "n_train": 200, "sampling": "first", "seed": SEED})
run.set_results(per_user=pd.DataFrame({f: r.eers for f, r in OPEN.items()}).assign(user=U.users_of(df)),
                metrics={"table": openset_table}, tables={"openset_table": openset_table},
                index={"mean_eer": openset_table.loc[0, "mean_eer"], "model": "table (best: " + openset_table.loc[0, "model"] + ")"})
openset_table

### 11c. Enrollment-size table

Mean EER per model, by sampling scheme (rows) and $n$ (columns). If `stratified` beats `first` by a wide
margin at the same $n$, session drift, not the amount of data, is what limits one-sitting enrollment.

In [ ]:
enroll_table = ENROLL.groupby(["model", "sampling", "n"])["eer"].mean().unstack("n")
run = start_run("table_enrollment", {"model": "all", "features": "log21", "protocol": "enrollment size",
                                     "split_short": "test", "n_train": list(ENROLL_N), "sampling": "first+stratified", "seed": SEED})
run.set_results(per_user=ENROLL, metrics={"table": enroll_table.reset_index()},
                tables={"enrollment_table": enroll_table.reset_index()})
enroll_table.round(4)

### 11d. Template-adaptation table

`genuine_accept_rate` = share of the 150 genuine logins that were accepted, and therefore added.
`impostors_absorbed` = total number of impostor typings that entered templates, out of 50 per subject.
`p_adapted_vs_static` = Wilcoxon p-value of adapted vs static.

In [ ]:
adapt_table = ADAPT.groupby(["n0", "model"]).agg(
    eer_static=("eer_static", "mean"), eer_adapted=("eer_adapted", "mean"), eer_oracle=("eer_oracle", "mean"),
    genuine_accept_rate=("genuine_accepted", lambda v: v.sum() / (150 * len(v))),
    impostors_absorbed=("impostors_absorbed", "sum"),
    users_with_absorbed=("impostors_absorbed", lambda v: int((v > 0).sum()))).reset_index()
adapt_table["p_adapted_vs_static"] = [U.paired_test(g.eer_adapted, g.eer_static)["wilcoxon_p"]
                                      for _, g in ADAPT.groupby(["n0", "model"])]
run = start_run("table_adaptation", {"model": ADAPT_FAMS, "features": "log21", "protocol": "template adaptation",
                                     "split_short": "adapt->test", "n_train": list(ADAPT_N0), "sampling": "first",
                                     "seed": SEED, "impostor_attempts_per_user": 50})
run.add_figure(P.adaptation_bars(adapt_table), "template_adaptation", report=True)
run.set_results(per_user=ADAPT, metrics={"table": adapt_table}, tables={"adaptation_table": adapt_table})
adapt_table.round(4)

### 11e. Speed-normalization table

Paired tests of (b) rhythm20 and (c) rhythm20+mean against (a) log21 for each model, at a realistic
one-sitting enrollment (n = 24, or the smallest n in quick mode) and at full enrollment (n = 200).
Negative `diff` = the speed-normalized features are better.

In [ ]:
speed_table = SPEED.groupby(["model", "features", "n"]).eer.mean().unstack("n")
N_CHECK = (24 if 24 in ENROLL_N else min(ENROLL_N), 200)
rows = []
for fam in SPEED_FAMS:
    for n in N_CHECK:
        a = SPEED[(SPEED.model == fam) & (SPEED.features == "log21") & (SPEED.n == n)].sort_values("user").eer.to_numpy()
        for fs in SPEED_FEATS[1:]:
            b = SPEED[(SPEED.model == fam) & (SPEED.features == fs) & (SPEED.n == n)].sort_values("user").eer.to_numpy()
            p = U.paired_test(b, a)
            rows.append({"model": fam, "n": n, "features": fs, "eer": b.mean(), "eer_log21": a.mean(), "diff": p["mean_diff"],
                         "diff_ci95": f"[{p['diff_ci95_low']:+.4f}, {p['diff_ci95_high']:+.4f}]", "wilcoxon_p": p["wilcoxon_p"]})
speed_tests = pd.DataFrame(rows)
run = start_run("table_speed", {"model": SPEED_FAMS, "features": list(SPEED_FEATS), "protocol": "first-n enrollment",
                                "split_short": "test", "n_train": list(ENROLL_N), "sampling": "first", "seed": SEED})
run.add_figure(P.speed_curves(SPEED, SPEED_FEATS, SPEED_FAMS, ENROLL_N), "speed_normalization", report=True)
run.set_results(per_user=SPEED, metrics={"table": speed_table.reset_index()},
                tables={"speed_table": speed_table.reset_index(), "speed_paired_tests": speed_tests})
speed_tests.round(4)

### 11f. GMM diagnostics: do components align with sessions?

For subjects where BIC picks $K \ge 2$, components are cross-tabulated against the 4 training sessions.
* ARI near 0 means the components are unrelated to sessions.
* Small minor-component weights mean a component that models a handful of unusual typings.

In [ ]:
gmms = OFFICIAL["GMM"].models
run = start_run("gmm_diagnostics", {"model": "GMM", "key_settings": TUNED["GMM"][0], "features": "log21",
                                    "protocol": "official (training typings 1-200)", "split_short": "train",
                                    "n_train": 200, "sampling": "first", "seed": SEED})
with run.capture():
    ks = pd.Series({u: m.k_ for u, m in gmms.items()})
    print("K chosen by BIC:", ks.value_counts().sort_index().to_dict())
    gdiag, ctabs = E.gmm_session_alignment(X, gmms)
    if len(gdiag):
        print("mean ARI vs session:", round(gdiag.ari_vs_session.mean(), 3),
              "| mean ARI vs early(1-2)/late(3-4):", round(gdiag.ari_vs_early_late.mean(), 3))
if len(gdiag):
    run.add_figure(P.gmm_components_vs_sessions(ctabs, gdiag, gmms), "gmm_components_vs_sessions", report=True)
run.set_results(per_user=gdiag if len(gdiag) else None,
                metrics={"k_counts": ks.value_counts().to_dict(),
                         "mean_ari_vs_session": float(gdiag.ari_vs_session.mean()) if len(gdiag) else None},
                index={"mean_eer": float(OFFICIAL["GMM"].eers.mean())})
gdiag.round(3) if len(gdiag) else "no subject with K >= 2"

### 11g. Summary

In [ ]:
print(f"total time {(time.time() - T0) / 60:.1f} min")
display(Markdown("**Official protocol**"))
display(official_table[["model", "val_eer", "mean_eer", "ci95", "test_ll_median", "diff_vs_base", "diff_ci95", "wilcoxon_p"]].round(4))
display(Markdown("**Open-set**"))
display(openset_table[["model", "mean_eer", "official_mean_eer", "ci95", "diff_vs_base", "wilcoxon_p"]].round(4))
display(Markdown("**Enrollment (first n)**"))
display(enroll_table.xs("first", level="sampling").round(4))

## 12. Save everything

For every run: `best_model/`, `visualizations/`, `metrics.json`, `per_user.csv`, `scores.npz` and `log.txt`.
Each run is also appended to `runs_index.csv`, and the report figures are copied to `report_figures/`.

In [ ]:
saved = save_all()
print(f"saved {len(saved)} runs to {saver.out}")
saved